In [14]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [15]:

import pandas as pd
import numpy as  np
import datetime as dt
import pandas as pd
import numpy as np
import datetime as dt

from Python_arq import engines as engs
from sqlalchemy import text
from calendar import day_name
from sqlalchemy import text
from pathlib import Path




In [16]:
eng = engs.get_engine()
text_qry = text(engs.load_query('qry_olos.sql'))
df = pd.read_sql(text_qry, eng)

In [17]:
dia_semana = {
    'Monday': 'Segunda',
    'Tuesday': 'Terça',
    'Wednesday': 'Quarta',
    'Thursday': 'Quinta',
    'Friday': 'Sexta',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df['day_week'] = df['data'].apply(lambda x: day_name[x.weekday()]).map(dia_semana)
df['data'] = pd.to_datetime(df['data'])
df['semana_mes'] = (
    df['data'].dt.day.sub(1) // 7 + 1 
)
df['day_week'] = df['day_week'] + '_W' + df['semana_mes'].astype(str)

In [18]:
## performance bases | Hoje ## 
df_today = df[df['data'].dt.date == dt.datetime.today().date()]
df_today = df_today.groupby(['area','base_type']).agg(
    Total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_today['performance'] = (df_today['total_atendidas'] / df_today['Total_tentativas'] * 100).round(2)
df_today = df_today.sort_values('performance', ascending=False)
df_today

,area,base_type,Total_tentativas,total_atendidas,performance


In [19]:
## top10 bases ## 
df_top10 = df.groupby(['area','base_type']).agg(
    total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_top10 = df_top10[df_top10['total_tentativas']>=1000]
df_top10['performance'] = (df_top10['total_atendidas'] / df_top10['total_tentativas'] * 100).round(2)
df_top10 = df_top10.sort_values('performance', ascending=False).head(10)
df_top10

,area,base_type,total_tentativas,total_atendidas,performance
28,Multi,evento,10897,467,4.29
26,Multi,Material Rico,3556,149,4.19
37,Pediatria,Base Leads,3153,131,4.15
41,Pediatria,evento,4805,189,3.93
5,Enfermagem,evento,30430,1110,3.65
0,Enfermagem,Base Leads,22737,782,3.44
7,Enfermagem,legado,13305,448,3.37
8,Fisioterapia,Base Leads,34216,1141,3.33
48,Psicologia,evento,17812,584,3.28
33,Nutrição,evento,24514,790,3.22


In [20]:
## performance bases | D-1 ## 
df_ontem = df[df['data'] == '2026-04-14']
df_ontem = df_ontem.groupby(['area','base_type']).agg(
    Total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_ontem['performance'] = (df_ontem['total_atendidas'] / df_ontem['Total_tentativas'] * 100).round(2)
df_ontem = df_ontem.sort_values('performance', ascending=False)
df_ontem

,area,base_type,Total_tentativas,total_atendidas,performance
0,Enfermagem,Material Rico,1406,39,2.77
3,Fisioterapia,evento,1717,44,2.56
4,Medicina,Base Leads,2235,48,2.15
9,Psicologia,inativa,722,15,2.08
1,Fisioterapia,Material Rico,4665,93,1.99
8,Psicologia,ativa,1011,16,1.58
6,Medicina,ativa,4291,63,1.47
2,Fisioterapia,ativa,3451,49,1.42
7,Psicologia,Material Rico,2211,31,1.40
5,Medicina,Material Rico,2957,41,1.39


In [21]:
bases_atuadas = df.groupby(['area','base_type','tablename','data']).agg(
    tentativas = ('tentativas','sum'),
    atendidas = ('atendidas','sum')
).reset_index()
bases_atuadas['split_tablename'] = bases_atuadas['tablename'].str.split('_')
bases_atuadas = bases_atuadas.to_excel(r'C:\Users\wconceicao\OneDrive - Grupo A Educação SA\Área de Trabalho\bases_atuadas.xlsx')

In [22]:
df = df.to_excel(r'C:\Users\wconceicao\OneDrive - Grupo A Educação SA\Área de Trabalho\tentativas.xlsx')